In [1]:
onto_context = [
  {
    "id": "business_concept_customer",
    "type": "business_concept",
    "name": "Customer",
    "aliases": ["paying customer", "purchased contact", "buyer"],
    "definition": "A Customer is a Contact that has at least one valid purchase, represented in the schema as an Order or SalesOrder.",
    "base_entities": ["Contact", "Order", "SalesOrder"],
    "business_rules": [
      "The Contact must be associated with at least one Order/SalesOrder.",
      "The Order/SalesOrder must have a valid (completed/paid) status."
    ],
    "derivation_logic": "Contact + at least one valid Order/SalesOrder = Customer",
    "graph_mapping": {
      "primary_node": "Contact",
      "related_nodes": ["Order", "SalesOrder"],
      "relationship_path": [
        "(Contact)-[:PLACED]->(Order)",
        "(SalesOrder)-[:PLACED_BY]->(Contact)"
      ]
    },
    "examples": [
      "A contact with no Orders is not considered a Customer.",
      "A contact with one completed SalesOrder is considered a Customer."
    ],
    "search_text": "Customer means a Contact with at least one valid Order or SalesOrder.",
    "schema_note": "Schema has no generic 'Purchase' node; Order (simple) and SalesOrder (full CPQ flow) both represent a purchase event. Use whichever your data actually populates."
  },
  {
    "id": "business_concept_user",
    "type": "business_concept",
    "name": "User",
    "aliases": ["employee", "staff member", "internal user"],
    "definition": "A User is an internal employee of a Tenant who operates the system, holds a Role, and belongs to a Team. A User is distinct from a Contact/Customer.",
    "base_entities": ["Tenant", "User", "Role", "Team"],
    "business_rules": [
      "A User must be employed by exactly one Tenant.",
      "A User must have at least one assigned Role.",
      "A User may optionally belong to a Team and follow BusinessHours."
    ],
    "derivation_logic": "Tenant + employed account + assigned Role = User",
    "graph_mapping": {
      "primary_node": "User",
      "related_nodes": ["Tenant", "Role", "Team", "BusinessHours", "Location"],
      "relationship_path": [
        "(Tenant)-[:EMPLOYS]->(User)",
        "(User)-[:HAS_ROLE]->(Role)",
        "(User)-[:MEMBER_OF]->(Team)",
        "(User)-[:FOLLOWS_HOURS]->(BusinessHours)",
        "(User)-[:LOCATED_AT]->(Location)"
      ]
    },
    "examples": [
      "An employee assigned an Admin Role and placed on the Support Team is a User.",
      "A Contact who buys the product is never a User."
    ],
    "search_text": "User means an internal employee of a Tenant with an assigned Role and Team, distinct from Contact/Customer."
  },
  {
    "id": "business_concept_support_agent",
    "type": "business_concept",
    "name": "Support Agent",
    "aliases": ["agent", "customer service rep", "CSR"],
    "definition": "A Support Agent is the schema's Agent node — a work-handling identity tied to a User — that carries Skills, Capacity, and Presence, and is assigned as a member of one or more Queues to handle Conversations.",
    "base_entities": ["Agent", "User", "Queue"],
    "business_rules": [
      "An Agent must be linked to exactly one underlying User via IS_AGENT_FOR.",
      "An Agent must be a member of at least one Queue to receive routed Conversations.",
      "An Agent's availability is governed by its AgentPresence and AgentCapacity."
    ],
    "derivation_logic": "User + Agent record (IS_AGENT_FOR) + Queue membership = Support Agent",
    "graph_mapping": {
      "primary_node": "Agent",
      "related_nodes": ["User", "Queue", "Skill", "AgentCapacity", "AgentPresence"],
      "relationship_path": [
        "(Agent)-[:IS_AGENT_FOR]->(User)",
        "(Queue)-[:HAS_MEMBER]->(Agent)",
        "(Agent)-[:HAS_SKILL]->(Skill)",
        "(Agent)-[:HAS_CAPACITY]->(AgentCapacity)",
        "(Agent)-[:HAS_PRESENCE]->(AgentPresence)"
      ]
    },
    "examples": [
      "A User enabled as an Agent and made a member of the Billing Queue is a Support Agent.",
      "A User with no Agent record cannot be routed a Conversation as a handler."
    ],
    "search_text": "Support Agent means the schema's Agent node linked to a User, holding Skills and Presence, assigned to a Queue.",
    "schema_note": "Your schema separates User (identity/employment) from Agent (work-handling role) via IS_AGENT_FOR — this is a cleaner version of what was previously modeled as 'User with Support Agent role.'"
  },
  {
    "id": "business_concept_sentiment_score",
    "type": "business_concept",
    "name": "Sentiment Score",
    "aliases": ["sentiment rating", "tone score"],
    "definition": "A Sentiment Score is a derived value representing the emotional tone of a Conversation, expected to be carried as a property on ConversationSummary or as a related Insight for the Contact.",
    "base_entities": ["Conversation", "ConversationSummary", "Insight"],
    "business_rules": [
      "The Conversation must have an associated ConversationSummary.",
      "The sentiment value is computed from Conversation/transcript content.",
      "Where tracked at the Contact level over time, it surfaces via Insight."
    ],
    "derivation_logic": "Conversation + ConversationSummary analysis (or Contact-level Insight) = Sentiment Score",
    "graph_mapping": {
      "primary_node": "Conversation",
      "related_nodes": ["ConversationSummary", "Insight", "Contact"],
      "relationship_path": [
        "(Conversation)-[:HAS_SUMMARY]->(ConversationSummary)",
        "(Contact)-[:HAS_INSIGHT]->(Insight)"
      ]
    },
    "examples": [
      "A ConversationSummary flagging frustrated language implies a negative Sentiment Score.",
      "A Contact-level Insight aggregating sentiment across Conversations shows an overall trend."
    ],
    "search_text": "Sentiment Score means the emotional tone value derived from a Conversation's summary or a Contact-level Insight.",
    "schema_note": "No dedicated SentimentScore node exists in the schema — model this as a property on ConversationSummary (per-conversation) and/or Insight (aggregated per-Contact)."
  },
  {
    "id": "business_concept_billing_cycle",
    "type": "business_concept",
    "name": "Billing Cycle",
    "aliases": ["billing period", "invoice cycle"],
    "definition": "A Billing Cycle is the recurring time interval, expected as a property on Contract or Invoice, during which charges are accrued and invoiced to a Company.",
    "base_entities": ["Contract", "Invoice", "Company"],
    "business_rules": [
      "The Contract must specify a billing frequency property (e.g., monthly, annual).",
      "Each Invoice generated must fall within a defined cycle window.",
      "Charges outside an active Contract's term are not part of a Billing Cycle."
    ],
    "derivation_logic": "Contract billing-frequency property + Invoice issuance dates = Billing Cycle",
    "graph_mapping": {
      "primary_node": "Contract",
      "related_nodes": ["Invoice", "Company"],
      "relationship_path": [
        "(Contract)-[:FOR_COMPANY]->(Company)",
        "(Invoice)-[:FOR_ORDER]->(SalesOrder)",
        "(Invoice)-[:ISSUED_TO]->(Company)"
      ]
    },
    "examples": [
      "A monthly-billed Contract produces a new Invoice to the Company every 30 days.",
      "An annual Contract has a single Billing Cycle per year."
    ],
    "search_text": "Billing Cycle means the recurring invoicing interval, tracked as a property on Contract and reflected in Invoice issuance to a Company.",
    "schema_note": "No dedicated BillingCycle node — treat as a property on Contract (frequency) and infer actual cycles from Invoice issue dates."
  },
  {
    "id": "business_concept_data_retention_policy",
    "type": "business_concept",
    "name": "Data Retention Policy",
    "aliases": ["retention rule", "data lifecycle policy"],
    "definition": "A Data Retention Policy is a governance rule defining how long Conversation, Transcript, and PrivacyRequest-related data must be stored before deletion or archival.",
    "base_entities": ["PrivacyRequest", "Conversation"],
    "business_rules": [
      "The policy must specify a retention duration per data type.",
      "A PrivacyRequest (e.g., deletion request) must be resolvable against the policy.",
      "Expired data must be flagged for archival or deletion."
    ],
    "derivation_logic": "Tenant-level retention rule + PrivacyRequest handling + Conversation data age = Data Retention Policy",
    "graph_mapping": {
      "primary_node": "PrivacyRequest",
      "related_nodes": ["Contact", "Conversation"],
      "relationship_path": [
        "(PrivacyRequest)-[:FOR_CONTACT]->(Contact)",
        "(PrivacyRequest)-[:REVIEWED_BY]->(User)"
      ]
    },
    "examples": [
      "A rule requiring Conversation data to be purged after 24 months is a Data Retention Policy.",
      "A PrivacyRequest for deletion that isn't honored within the policy window is a compliance gap."
    ],
    "search_text": "Data Retention Policy means the governance rule for how long Conversation data is retained, enforced and tracked via PrivacyRequest.",
    "schema_note": "No dedicated DataRetentionPolicy node in your schema — PrivacyRequest is the closest existing node (handles deletion/access requests). Consider adding a Tenant-level policy node if you need to store the retention duration itself."
  },
  {
    "id": "business_concept_lead",
    "type": "business_concept",
    "name": "Lead",
    "aliases": ["marketing lead", "inbound lead"],
    "definition": "A Lead is a Contact that has an associated LeadSummary or LeadEnrichmentSnapshot but has not yet been converted to a Deal.",
    "base_entities": ["Contact", "LeadSummary", "LeadEnrichmentSnapshot", "ContactProfile"],
    "business_rules": [
      "The Contact must have a LeadSummary and/or LeadEnrichmentSnapshot.",
      "The associated ContactProfile must not yet have CONVERTED_TO a Deal.",
      "A Lead has no associated Contract."
    ],
    "derivation_logic": "Contact + LeadSummary/LeadEnrichmentSnapshot + no Deal conversion = Lead",
    "graph_mapping": {
      "primary_node": "Contact",
      "related_nodes": ["LeadSummary", "LeadEnrichmentSnapshot", "ContactProfile"],
      "relationship_path": [
        "(Contact)-[:HAS_LEAD_SUMMARY]->(LeadSummary)",
        "(Contact)-[:HAS_ENRICHMENT]->(LeadEnrichmentSnapshot)",
        "(ContactProfile)-[:OVERLAY_OF]->(Contact)"
      ]
    },
    "examples": [
      "A contact with an enrichment snapshot but no ContactProfile→Deal conversion is a Lead.",
      "A contact whose ContactProfile has CONVERTED_TO a Deal is no longer just a Lead."
    ],
    "search_text": "Lead means a Contact with a LeadSummary or LeadEnrichmentSnapshot that has not yet converted to a Deal."
  },
  {
    "id": "business_concept_prospect",
    "type": "business_concept",
    "name": "Prospect",
    "aliases": ["qualified lead", "sales prospect"],
    "definition": "A Prospect is a Lead (Contact) whose ContactProfile has been converted into a Deal, owned by a User and linked to a Company/CompanySite.",
    "base_entities": ["Contact", "ContactProfile", "Deal", "User"],
    "business_rules": [
      "The Contact's ContactProfile must have CONVERTED_TO a Deal.",
      "The Deal must be OWNED_BY a User.",
      "The Contact must not yet have a signed Contract/SalesOrder tied to that Deal."
    ],
    "derivation_logic": "Lead (Contact) + ContactProfile CONVERTED_TO Deal = Prospect",
    "graph_mapping": {
      "primary_node": "Contact",
      "related_nodes": ["ContactProfile", "Deal", "User"],
      "relationship_path": [
        "(ContactProfile)-[:OVERLAY_OF]->(Contact)",
        "(ContactProfile)-[:CONVERTED_TO]->(Deal)",
        "(Deal)-[:PRIMARY_CONTACT]->(Contact)",
        "(Deal)-[:OWNED_BY]->(User)"
      ]
    },
    "examples": [
      "A contact whose profile converts to an open Deal owned by a sales rep is a Prospect.",
      "A lead with no ContactProfile conversion remains just a Lead."
    ],
    "search_text": "Prospect means a Lead (Contact) whose ContactProfile has converted into a Deal owned by a User."
  },
  {
    "id": "business_concept_contract",
    "type": "business_concept",
    "name": "Contract",
    "aliases": ["agreement", "service agreement"],
    "definition": "A Contract is a formal, binding agreement for a Company, originating from a Deal, defining terms of purchase and renewal.",
    "base_entities": ["Contract", "Company", "Deal"],
    "business_rules": [
      "The Contract must be linked to a Company via FOR_COMPANY.",
      "The Contract must originate FROM_DEAL.",
      "A renewal Contract links back to a new Deal via RENEWED_BY."
    ],
    "derivation_logic": "Deal + won outcome + Company terms = Contract",
    "graph_mapping": {
      "primary_node": "Contract",
      "related_nodes": ["Company", "Deal"],
      "relationship_path": [
        "(Tenant)-[:HAS_CONTRACT]->(Contract)",
        "(Contract)-[:FOR_COMPANY]->(Company)",
        "(Contract)-[:FROM_DEAL]->(Deal)",
        "(Contract)-[:RENEWED_BY]->(Deal)"
      ]
    },
    "examples": [
      "A signed annual agreement tied to a closed-won Deal is a Contract.",
      "A Contract nearing expiry that is RENEWED_BY a new Deal starts a renewal cycle."
    ],
    "search_text": "Contract means a binding agreement for a Company originating from a Deal, with support for renewal via a new Deal."
  },
  {
    "id": "business_concept_sales_pipeline",
    "type": "business_concept",
    "name": "Sales Pipeline",
    "aliases": ["pipeline", "deal pipeline"],
    "definition": "A Sales Pipeline is the overall set of open Deals owned by Users, tracked from creation through DealActivity to a won Contract or lost outcome.",
    "base_entities": ["Deal", "User", "DealActivity"],
    "business_rules": [
      "Each Deal must be OWNED_BY a User.",
      "Deal progress is tracked through associated DealActivity records.",
      "A Deal exits the pipeline as either FROM_DEAL on a Contract (won) or marked lost."
    ],
    "derivation_logic": "Set of Deals owned by Users + tracked DealActivity progression = Sales Pipeline",
    "graph_mapping": {
      "primary_node": "Deal",
      "related_nodes": ["User", "DealActivity", "Contract"],
      "relationship_path": [
        "(Tenant)-[:HAS_DEAL]->(Deal)",
        "(Deal)-[:OWNED_BY]->(User)",
        "(Deal)-[:HAS_ACTIVITY]->(DealActivity)",
        "(Contract)-[:FROM_DEAL]->(Deal)"
      ]
    },
    "examples": [
      "All open Deals owned by the sales team make up the Sales Pipeline.",
      "A Deal with no activity for 90 days is a stalled pipeline entry."
    ],
    "search_text": "Sales Pipeline means the set of open Deals, owned by Users and tracked via DealActivity, moving toward a won Contract.",
    "schema_note": "Your schema has no explicit Deal 'stage' node — treat pipeline stage as a property on Deal itself, or add a DealStage node if you need formal stage-gate reporting (see Sales Stage below)."
  },
  {
    "id": "business_concept_sales_stage",
    "type": "business_concept",
    "name": "Sales Stage",
    "aliases": ["deal stage", "pipeline stage"],
    "definition": "A Sales Stage is the current status of a Deal (e.g., Qualification, Negotiation, Closed-Won) as it progresses toward becoming a Contract.",
    "base_entities": ["Deal"],
    "business_rules": [
      "Every Deal must have a current stage value at all times.",
      "Stage changes should be reflected in DealActivity for auditability.",
      "The terminal stages are Closed-Won (leads to Contract) or Closed-Lost."
    ],
    "derivation_logic": "Deal + current stage property + DealActivity history = Sales Stage",
    "graph_mapping": {
      "primary_node": "Deal",
      "related_nodes": ["DealActivity"],
      "relationship_path": [
        "(Deal)-[:HAS_ACTIVITY]->(DealActivity)"
      ]
    },
    "examples": [
      "A Deal moving from 'Negotiation' to 'Closed-Won' changes its Sales Stage property.",
      "A DealActivity entry logging a stage change provides an audit trail."
    ],
    "search_text": "Sales Stage means the current status property of a Deal as it progresses, logged via DealActivity.",
    "schema_note": "No dedicated stage/pipeline-step node exists — Sales Stage should be modeled as a property on Deal unless you want a separate DealStage lookup node for a fixed, ordered list of stages."
  },
  {
    "id": "business_concept_lead_scoring",
    "type": "business_concept",
    "name": "Lead Scoring",
    "aliases": ["lead score", "propensity score"],
    "definition": "Lead Scoring is a derived numeric value on a Contact's CustomerScore, indicating likelihood of conversion, informed by LeadEnrichmentSnapshot and engagement.",
    "base_entities": ["Contact", "CustomerScore", "LeadEnrichmentSnapshot"],
    "business_rules": [
      "The Contact must have a CustomerScore record.",
      "The score should incorporate signals from LeadEnrichmentSnapshot where available.",
      "Lead Scoring thresholds determine progression to MQL/SQL-equivalent status."
    ],
    "derivation_logic": "Contact + CustomerScore + LeadEnrichmentSnapshot signals = Lead Scoring",
    "graph_mapping": {
      "primary_node": "Contact",
      "related_nodes": ["CustomerScore", "LeadEnrichmentSnapshot"],
      "relationship_path": [
        "(Contact)-[:HAS_SCORES]->(CustomerScore)",
        "(Contact)-[:HAS_ENRICHMENT]->(LeadEnrichmentSnapshot)"
      ]
    },
    "examples": [
      "A Contact with a high CustomerScore and rich enrichment data has strong Lead Scoring.",
      "A Contact with no CustomerScore record has no computed Lead Scoring."
    ],
    "search_text": "Lead Scoring means the numeric value on a Contact's CustomerScore, informed by LeadEnrichmentSnapshot signals."
  },
  {
    "id": "business_concept_mql",
    "type": "business_concept",
    "name": "Marketing Qualified Lead (MQL)",
    "aliases": ["MQL"],
    "definition": "A Marketing Qualified Lead is a Contact whose CustomerScore has crossed a marketing-defined threshold, prior to ContactProfile conversion into a Deal.",
    "base_entities": ["Contact", "CustomerScore"],
    "business_rules": [
      "The Contact's CustomerScore must be at or above the marketing qualification threshold.",
      "The Contact's ContactProfile must not yet have CONVERTED_TO a Deal."
    ],
    "derivation_logic": "Contact + CustomerScore >= marketing threshold + no Deal conversion = MQL",
    "graph_mapping": {
      "primary_node": "Contact",
      "related_nodes": ["CustomerScore", "ContactProfile"],
      "relationship_path": [
        "(Contact)-[:HAS_SCORES]->(CustomerScore)",
        "(ContactProfile)-[:OVERLAY_OF]->(Contact)"
      ]
    },
    "examples": [
      "A Contact scoring above the marketing threshold, not yet converted, is an MQL.",
      "A Contact below threshold remains a raw Lead."
    ],
    "search_text": "MQL means a Contact whose CustomerScore has crossed the marketing threshold, prior to Deal conversion.",
    "schema_note": "No dedicated MQL flag exists — represent as a threshold rule evaluated against CustomerScore, or add a status property/enum on ContactProfile (e.g., lead_status = 'MQL')."
  },
  {
    "id": "business_concept_sql",
    "type": "business_concept",
    "name": "Sales Qualified Lead (SQL)",
    "aliases": ["SQL"],
    "definition": "A Sales Qualified Lead is a Contact whose ContactProfile has been formally CONVERTED_TO a Deal, accepted and owned by a User.",
    "base_entities": ["Contact", "ContactProfile", "Deal", "User"],
    "business_rules": [
      "The Contact must have previously qualified as an MQL.",
      "The ContactProfile must have CONVERTED_TO a Deal.",
      "The resulting Deal must be OWNED_BY a User (sales rep)."
    ],
    "derivation_logic": "MQL (Contact) + ContactProfile CONVERTED_TO Deal owned by User = SQL",
    "graph_mapping": {
      "primary_node": "Contact",
      "related_nodes": ["ContactProfile", "Deal", "User"],
      "relationship_path": [
        "(ContactProfile)-[:OVERLAY_OF]->(Contact)",
        "(ContactProfile)-[:CONVERTED_TO]->(Deal)",
        "(Deal)-[:OWNED_BY]->(User)"
      ]
    },
    "examples": [
      "An MQL whose profile converts into a Deal owned by a rep becomes an SQL.",
      "An MQL rejected by sales, with no Deal created, does not become an SQL."
    ],
    "search_text": "SQL means a Contact whose ContactProfile has formally converted into a Deal owned by a User."
  },
  {
    "id": "business_concept_upsell_cross_sell_suggestion",
    "type": "business_concept",
    "name": "Upsell/Cross-sell Suggestion",
    "aliases": ["upsell recommendation", "cross-sell offer"],
    "definition": "An Upsell/Cross-sell Suggestion is a Recommendation generated for a Contact, typically pointing to an Addon, ProductOffer, or upgraded Product relative to their existing Contract/SalesOrder.",
    "base_entities": ["Contact", "Recommendation", "Addon", "ProductOffer", "Product"],
    "business_rules": [
      "The Recommendation must target a Contact with an existing Contract or SalesOrder.",
      "The Recommendation should reference an Addon, ProductOffer, or Product the Contact does not already have.",
      "The suggestion must differ from the Contact's current Product/Addon set."
    ],
    "derivation_logic": "Existing Contract/SalesOrder + Contact-[:HAS_RECOMMENDATION]->Recommendation targeting Addon/ProductOffer = Upsell/Cross-sell Suggestion",
    "graph_mapping": {
      "primary_node": "Recommendation",
      "related_nodes": ["Contact", "Addon", "ProductOffer", "Product"],
      "relationship_path": [
        "(Contact)-[:HAS_RECOMMENDATION]->(Recommendation)",
        "(Addon)-[:FOR_PRODUCT]->(Product)",
        "(ProductOffer)-[:FOR_PRODUCT]->(Product)"
      ]
    },
    "examples": [
      "Recommending a premium Addon to a Contact who frequently hits usage limits is an Upsell Suggestion.",
      "Recommending an unrelated Product to a new Lead with no Contract is not an Upsell/Cross-sell Suggestion."
    ],
    "search_text": "Upsell/Cross-sell Suggestion means a Recommendation for a Contact pointing to an Addon, ProductOffer, or upgraded Product."
  },
  {
    "id": "business_concept_nps",
    "type": "business_concept",
    "name": "Net Promoter Score (NPS)",
    "aliases": ["NPS"],
    "definition": "Net Promoter Score is a derived metric from Contact survey responses collected post-Conversation, measuring likelihood of the Contact to recommend the organization.",
    "base_entities": ["Conversation", "Contact", "DispositionEvent"],
    "business_rules": [
      "The Conversation must have an associated post-interaction survey response from the Contact.",
      "Respondents must be categorized as Promoters, Passives, or Detractors based on their score.",
      "NPS is calculated as percentage of Promoters minus percentage of Detractors."
    ],
    "derivation_logic": "% Promoters - % Detractors from post-Conversation Contact survey responses = NPS",
    "graph_mapping": {
      "primary_node": "Conversation",
      "related_nodes": ["Contact", "DispositionEvent"],
      "relationship_path": [
        "(Contact)-[:PARTICIPATED_IN]->(Conversation)",
        "(Conversation)-[:HAS_DISPOSITION_EVENT]->(DispositionEvent)"
      ]
    },
    "examples": [
      "60% Promoters and 10% Detractors results in an NPS of +50.",
      "A survey with no responses cannot generate an NPS value."
    ],
    "search_text": "NPS means the survey-derived score measuring a Contact's likelihood to recommend, calculated as Promoters minus Detractors.",
    "schema_note": "No dedicated Survey/NPS node exists — model the raw score as a property on DispositionEvent (or a new SurveyResponse node) rather than assuming a built-in survey entity."
  },
  {
    "id": "business_concept_csat",
    "type": "business_concept",
    "name": "Customer Satisfaction (CSAT)",
    "aliases": ["CSAT", "satisfaction score"],
    "definition": "Customer Satisfaction is a derived metric from post-Conversation survey ratings given by a Contact, measuring how satisfied the Contact was with a specific interaction.",
    "base_entities": ["Conversation", "Contact", "DispositionEvent"],
    "business_rules": [
      "The Conversation must have an associated satisfaction survey response from the Contact.",
      "CSAT is typically calculated as the percentage of respondents rating the interaction positively.",
      "CSAT can be aggregated per Agent by joining through Conversation-[:HANDLED_BY]->Agent."
    ],
    "derivation_logic": "% of positive post-Conversation Contact survey ratings = CSAT",
    "graph_mapping": {
      "primary_node": "Conversation",
      "related_nodes": ["Contact", "DispositionEvent", "Agent"],
      "relationship_path": [
        "(Contact)-[:PARTICIPATED_IN]->(Conversation)",
        "(Conversation)-[:HAS_DISPOSITION_EVENT]->(DispositionEvent)",
        "(Conversation)-[:HANDLED_BY]->(Agent)"
      ]
    },
    "examples": [
      "85% of customers rating an interaction 4 or 5 stars gives an 85% CSAT.",
      "CSAT can be rolled up per Agent to compare individual performance."
    ],
    "search_text": "CSAT means the percentage of positive post-Conversation survey ratings given by a Contact, measuring interaction satisfaction.",
    "schema_note": "Same as NPS — no dedicated survey node; store the rating as a property on DispositionEvent or a new SurveyResponse node."
  },
  {
    "id": "business_concept_retention_churn_indicator",
    "type": "business_concept",
    "name": "Retention / Churn Indicator",
    "aliases": ["churn risk score", "retention signal"],
    "definition": "A Retention / Churn Indicator is a derived signal on a Contact's CustomerScore or Insight, based on Contract status and Conversation history, estimating likelihood to renew or cancel.",
    "base_entities": ["Contact", "CustomerScore", "Insight", "Contract"],
    "business_rules": [
      "The Contact must be linked (via Company) to an active or recently expired Contract.",
      "The indicator must incorporate CustomerScore and/or Insight signals from recent Conversations.",
      "The indicator must be expressed as a risk level or probability."
    ],
    "derivation_logic": "Contract status + Contact CustomerScore/Insight signals = Retention / Churn Indicator",
    "graph_mapping": {
      "primary_node": "Contact",
      "related_nodes": ["CustomerScore", "Insight", "Contract", "Company"],
      "relationship_path": [
        "(Contact)-[:HAS_SCORES]->(CustomerScore)",
        "(Contact)-[:HAS_INSIGHT]->(Insight)",
        "(Contract)-[:FOR_COMPANY]->(Company)"
      ]
    },
    "examples": [
      "A Contact with a declining CustomerScore near contract renewal shows a high churn risk.",
      "A Contact with a strong Insight trend shows a healthy Retention Indicator."
    ],
    "search_text": "Retention / Churn Indicator means a risk signal on a Contact's CustomerScore or Insight, combined with Contract status via Company."
  },
  {
    "id": "business_concept_customer_journey",
    "type": "business_concept",
    "name": "Customer Journey",
    "aliases": ["client journey", "end-to-end journey"],
    "definition": "A Customer Journey is the complete, ordered sequence of Interaction and Conversation records a Contact has with the organization across Queues and Channels.",
    "base_entities": ["Contact", "Interaction", "Conversation", "Channel"],
    "business_rules": [
      "The journey must include all Interaction records projected from a Contact's Conversations.",
      "Interactions must be ordered chronologically.",
      "The journey spans pre-sale (Deal/ContactActivity) and post-sale (Conversation/Interaction) touchpoints."
    ],
    "derivation_logic": "Contact + chronologically ordered Interaction/Conversation records = Customer Journey",
    "graph_mapping": {
      "primary_node": "Contact",
      "related_nodes": ["Interaction", "Conversation", "Channel"],
      "relationship_path": [
        "(Interaction)-[:FOR_CONTACT]->(Contact)",
        "(Interaction)-[:PROJECTED_FROM]->(Conversation)",
        "(Conversation)-[:THROUGH]->(Channel)"
      ]
    },
    "examples": [
      "A contact's ordered Interaction records across chat and phone Conversations form their Customer Journey.",
      "A single isolated Conversation with no broader Interaction history is not the full Customer Journey."
    ],
    "search_text": "Customer Journey means the complete chronological sequence of a Contact's Interaction and Conversation records across Channels.",
    "schema_note": "No dedicated ConversationJourney node exists — Interaction (which is PROJECTED_FROM Conversation) is the correct node to aggregate a Contact's full journey."
  },
  {
    "id": "business_concept_account",
    "type": "business_concept",
    "name": "Account",
    "aliases": ["organization record", "company record"],
    "definition": "An Account maps directly to the schema's Company node — the organizational entity that groups Contacts (via ContactProfile), CompanySites, Deals, and Contracts.",
    "base_entities": ["Company", "Contact", "ContactProfile"],
    "business_rules": [
      "A Company must have at least one associated ContactProfile (WORKS_AT).",
      "A Company may have one or more CompanySites.",
      "Deals and Contracts roll up to a Company for billing and reporting."
    ],
    "derivation_logic": "Company + linked ContactProfiles (WORKS_AT) + CompanySites = Account",
    "graph_mapping": {
      "primary_node": "Company",
      "related_nodes": ["ContactProfile", "CompanySite", "Deal", "Contract"],
      "relationship_path": [
        "(Tenant)-[:HAS_COMPANY]->(Company)",
        "(ContactProfile)-[:WORKS_AT]->(Company)",
        "(Company)-[:HAS_SITE]->(CompanySite)",
        "(Deal)-[:WITH_COMPANY]->(Company)",
        "(Contract)-[:FOR_COMPANY]->(Company)"
      ]
    },
    "examples": [
      "A B2B company with five employee ContactProfiles is represented as one Company/Account.",
      "A CompanySite represents a specific branch or location of that Account."
    ],
    "search_text": "Account means the schema's Company node, grouping ContactProfiles, CompanySites, Deals, and Contracts under one organization.",
    "schema_note": "This concept maps 1:1 onto the existing Company node — no new node needed. The previous version's proposed 'Account' node can be dropped in favor of Company."
  },
  {
    "id": "business_concept_contact_point",
    "type": "business_concept",
    "name": "Contact Point",
    "aliases": ["reachability channel", "contact method"],
    "definition": "A ContactPoint is a specific, verified means of reaching a Contact (phone, email, messaging handle), distinct from ChannelIdentity which represents platform-specific identifiers.",
    "base_entities": ["Contact", "ContactPoint", "ChannelIdentity"],
    "business_rules": [
      "A ContactPoint must be linked to exactly one Contact.",
      "A Contact may have multiple ContactPoints and ChannelIdentities.",
      "A ContactPoint must have a defined type (phone, email, messaging, etc.)."
    ],
    "derivation_logic": "Contact + verified reachable identifier (ContactPoint) or platform identity (ChannelIdentity) = Contact Point",
    "graph_mapping": {
      "primary_node": "ContactPoint",
      "related_nodes": ["Contact", "ChannelIdentity"],
      "relationship_path": [
        "(Contact)-[:HAS_CONTACT_POINT]->(ContactPoint)",
        "(Contact)-[:HAS_CHANNEL_IDENTITY]->(ChannelIdentity)"
      ]
    },
    "examples": [
      "A verified mobile number is a ContactPoint used for SMS and calls.",
      "A Contact's WhatsApp handle is tracked as a ChannelIdentity, distinct from their raw phone ContactPoint."
    ],
    "search_text": "ContactPoint means a specific verified means of reaching a Contact; ChannelIdentity is the related platform-specific identifier.",
    "schema_note": "This concept already exists as an explicit node (HAS_CONTACT_POINT) — matched exactly. Also linked ChannelIdentity as the closely related platform-identity node."
  },
  {
    "id": "business_concept_consent_opt_in_status",
    "type": "business_concept",
    "name": "Consent / Opt-in Status",
    "aliases": ["consent status", "communication preference"],
    "definition": "Consent / Opt-in Status is a compliance attribute indicating whether a Contact has agreed to be contacted via a given ContactPoint/Channel, tracked and auditable through PrivacyRequest.",
    "base_entities": ["Contact", "ContactPoint", "PrivacyRequest"],
    "business_rules": [
      "Each ContactPoint should carry an explicit consent status property (opted-in, opted-out, unknown).",
      "Outbound Conversations to a ContactPoint without valid consent are non-compliant.",
      "A PrivacyRequest can trigger a change to consent status and must be REVIEWED_BY a User."
    ],
    "derivation_logic": "ContactPoint consent property + PrivacyRequest history = Consent / Opt-in Status",
    "graph_mapping": {
      "primary_node": "ContactPoint",
      "related_nodes": ["Contact", "PrivacyRequest", "User"],
      "relationship_path": [
        "(Contact)-[:HAS_CONTACT_POINT]->(ContactPoint)",
        "(PrivacyRequest)-[:FOR_CONTACT]->(Contact)",
        "(PrivacyRequest)-[:REVIEWED_BY]->(User)"
      ]
    },
    "examples": [
      "A contact who opted in to SMS can legally be messaged on that ContactPoint.",
      "A PrivacyRequest to opt out, once reviewed, updates the ContactPoint's consent status."
    ],
    "search_text": "Consent / Opt-in Status means the compliance attribute tracking whether a Contact has agreed to be reached via a ContactPoint, managed through PrivacyRequest.",
    "schema_note": "No dedicated consent-status node — model it as a property on ContactPoint, with PrivacyRequest providing the auditable change/request trail."
  }
]

In [2]:
import meilisearch
client = meilisearch.Client('http://192.168.11.156:7701', 'iamgroot')

index = client.index('')

index.add_documents(onto_context, primary_key='id')


TaskInfo(task_uid=25, index_uid='', status='enqueued', type='documentAdditionOrUpdate', enqueued_at=datetime.datetime(2026, 8, 22, 11, 0, 55, 463874))

In [5]:
CONFIG = {
    "knowledge_graph": {
        "uri": "neo4j://192.168.11.25:7688",
        "user": "neo4j",
        "password": "India@321", 
        "db_name": "org1000"
    },
    "meilisearch": {
        "url": "http://192.168.11.156:7701",
        "api_key": "iamgroot"
    }
}


In [6]:
MEILI_CLIENT = None

def get_meilisearch_client(config: dict = None):
    global MEILI_CLIENT
    if MEILI_CLIENT is None and config:
        url = config.get('url', 'http://192.168.11.156:7701')
        api_key = config.get('api_key', 'iamgroot')
        MEILI_CLIENT = meilisearch.Client(url, api_key)
    return MEILI_CLIENT

# ------------------------------------Meilisearch query execution function------------------------------------------
def search_business_concepts(query: str, limit: int = 4) -> dict:
    try:
        config = CONFIG.get('meilisearch', {})
        client = get_meilisearch_client(config)
        if not client:
            return {"error": "Meilisearch client not initialized. Check CONFIG."}
        
        index = client.index('')
        search_results = index.search(query, {
            'limit': limit,
            'hybrid': {
                'embedder': 'default',
                'semanticRatio': 0.5
            }
        })
        
        return {
            "query": query,
            "hits": search_results.get('hits', []),
            "total_hits": search_results.get('estimatedTotalHits', 0)
        }
    except Exception as e:
        return {"error": str(e)}

In [8]:
client.index("").update_settings({
    "embedders": {
        "default": {
            "source": "huggingFace",
            "model": "sentence-transformers/all-MiniLM-L6-v2"
        }
    }
})

TaskInfo(task_uid=26, index_uid='', status='enqueued', type='settingsUpdate', enqueued_at=datetime.datetime(2026, 8, 22, 11, 28, 40, 84307))

In [9]:
search_business_concepts("what about the billing cycles ?")

{'query': 'what about the billing cycles ?',
 'hits': [{'id': 'business_concept_billing_cycle',
   'type': 'business_concept',
   'name': 'Billing Cycle',
   'aliases': ['billing period', 'invoice cycle'],
   'definition': 'A Billing Cycle is the recurring time interval, expected as a property on Contract or Invoice, during which charges are accrued and invoiced to a Company.',
   'base_entities': ['Contract', 'Invoice', 'Company'],
   'business_rules': ['The Contract must specify a billing frequency property (e.g., monthly, annual).',
    'Each Invoice generated must fall within a defined cycle window.',
    "Charges outside an active Contract's term are not part of a Billing Cycle."],
   'derivation_logic': 'Contract billing-frequency property + Invoice issuance dates = Billing Cycle',
   'graph_mapping': {'primary_node': 'Contract',
    'related_nodes': ['Invoice', 'Company'],
    'relationship_path': ['(Contract)-[:FOR_COMPANY]->(Company)',
     '(Invoice)-[:FOR_ORDER]->(SalesOrder